In [ ]:
import json

def get_results_from_slurm(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        txt = f.read()
        lines = txt.split("\n")
        
        results = [{
            "matthews": [],
            "eval_loss": [],
            "train_loss": []
        }]
        
        for line in lines:
            if len(line) > 0 and line[0] == "{":
                obj = json.loads(line.replace("'", '"'))
                
                if "eval_matthews_correlation" in obj:
                    results[-1]["matthews"].append(obj["eval_matthews_correlation"])
                    if "eval_loss" in obj:
                        results[-1]["eval_loss"].append(obj["eval_loss"])
                
                elif "loss" in obj:
                    results[-1]["train_loss"].append(obj["loss"])
            
            # Uncomment below to split into multiple runs if needed
            # elif len(line) > 0 and " Run " in line:
            #     results.append({"matthews": [], "eval_loss": [], "train_loss": []})
            
    return results

def compute_stats(values, label):
    if not values:
        return f"{label} - No data"
    l = len(values)
    mean = sum(values) / l
    sorted_vals = sorted(values)
    median = sorted_vals[l // 2] if l % 2 == 1 else (sorted_vals[(l - 1) // 2] + sorted_vals[(l + 1) // 2]) / 2
    best = min(values) if "loss" in label.lower() else max(values)
    best_idx = values.index(best)
    return f"{label}:\n  Mean: {mean:.4f}\n  Median: {median:.4f}\n  Best: {best:.4f} (EPOCH: {best_idx})"

def print_info_from_slurm_results(results):
    for i, run in enumerate(results, start=1):
        print(f"=== Run {i} =============")

        print(compute_stats(run["matthews"], "Matthews Corr"))
        print(compute_stats(run["eval_loss"], "Eval Loss"))
        print(compute_stats(run["train_loss"], "Train Loss"))

# Example usage
file_path_hypernet = ".\\outputs\\hypernet_cola.txt"
file_path_baseline = ".\\outputs\\baseline_cola.txt"

results_hypernet = get_results_from_slurm(file_path=file_path_hypernet)
results_baseline = get_results_from_slurm(file_path=file_path_baseline)

print("HYPERNET")
print_info_from_slurm_results(results_hypernet)

print("\nBASELINE")
print_info_from_slurm_results(results_baseline)


HYPERNET
=== Run 1 =============
Matthews Corr:
  Mean: 0.5525
  Median: 0.5622
  Best: 0.6100 (EPOCH: 48)
Eval Loss:
  Mean: 0.6860
  Median: 0.6968
  Best: 0.9371 (EPOCH: 71)
Train Loss:
  Mean: 0.1923
  Median: 0.1587
  Best: 0.5789 (EPOCH: 0)

BASELINE
=== Run 1 =============
Matthews Corr:
  Mean: 0.5771
  Median: 0.5882
  Best: 0.6132 (EPOCH: 48)
Eval Loss:
  Mean: 0.7860
  Median: 0.8333
  Best: 1.1205 (EPOCH: 58)
Train Loss:
  Mean: 0.1699
  Median: 0.1337
  Best: 0.5601 (EPOCH: 0)
